# General Block Forward Example

Define a block-structured general problem inside the notebook with `HighLevelNLPBuilder`, generate labels with SLSQP, then train KKT-HardNet directly.

In [ ]:
from pathlib import Path
import json
import time

import jax.numpy as jnp
import numpy as np

from jaxmodel import HighLevelNLPBuilder
from kkthn.problems import apply_label_noise, generate_labels_with_slsqp, write_json
from kkthn.projection import ProjectionSettings
from kkthn.training import KKTTrainConfig, train_kkt_hardnet


def make_run_dir(name):
    root = Path.cwd().resolve()
    notebook_dir = root if root.name == "notebooks" else root / "notebooks"
    run_dir = notebook_dir / "_runs" / name / time.strftime("%Y%m%d_%H%M%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def make_train_config(config, proj):
    return KKTTrainConfig(
        epochs=int(config["epochs"]),
        batch_size=int(config["batch_size"]),
        learning_rate=float(config["learning_rate"]),
        train_frac=float(config["train_frac"]),
        hidden_size=int(config["hidden_size"]),
        hidden_layers=int(config["hidden_layers"]),
        seed=int(config["seed"]),
        dtype=str(config["dtype"]),
        print_every=int(config["print_every"]),
        projection=ProjectionSettings(**proj),
    )


In [ ]:
DATA = {
    "type": "general_block",
    "num_samples": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "x_L": [-1.0, -1.0],
    "x_U": [1.0, 1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


In [ ]:
def build_block_problem(dtype=jnp.float64):
    n_x = 2
    n_y = 4
    q_diag = jnp.asarray([1.4, 1.2, 0.8, 0.9], dtype=dtype)

    def objective(params, vars_dict):
        del params
        y = vars_dict["y"]
        return 0.5 * jnp.sum(q_diag * y**2) + 0.05 * jnp.sum(jnp.sin(y))

    def equality_block(params, vars_dict):
        x = params["x"]
        y = vars_dict["y"]
        return jnp.asarray([
            y[0] + 0.10 * y[2] ** 2 - x[0],
            y[1] + 0.10 * y[3] ** 2 - x[1],
        ], dtype=dtype)

    def inequality_block(params, vars_dict):
        del params
        y = vars_dict["y"]
        return jnp.asarray([
            y[0] ** 2 + y[2] ** 2 - 3.0,
            y[1] ** 2 + y[3] ** 2 - 3.0,
        ], dtype=dtype)

    zeros = jnp.zeros((n_y, n_x), dtype=dtype)
    lower = -3.0 * jnp.ones((n_y,), dtype=dtype)
    upper = 3.0 * jnp.ones((n_y,), dtype=dtype)
    params0 = {"x": jnp.zeros((n_x,), dtype=dtype)}

    return (
        HighLevelNLPBuilder(dtype=dtype)
        .add_parameter("x", n_x)
        .add_variable("y", n_y)
        .set_objective(objective)
        .add_nonlinear_equality(equality_block, name="equality_block")
        .add_nonlinear_inequality(inequality_block, name="inequality_block")
        .set_affine_lower_bound(var_name="y", param_name="x", M=zeros, c=lower)
        .set_affine_upper_bound(var_name="y", param_name="x", M=zeros, c=upper)
        .build(example_params=params0, jit_compile=True)
    )


In [ ]:
run_dir = make_run_dir("general_block_forward")
model = build_block_problem()

rng = np.random.default_rng(DATA["seed"])
X = rng.uniform(DATA["x_L"], DATA["x_U"], size=(DATA["num_samples"], 2))
Y, label_metadata = generate_labels_with_slsqp(model, X)
Y, label_metadata = apply_label_noise(Y, DATA, label_metadata)
label_metadata.update({
    "problem_type": "general_block",
    "n_x": 2,
    "n_y": 4,
    "n_eq": 2,
    "n_ineq": 2,
})

write_json(run_dir / "data.json", DATA)
write_json(run_dir / "config.json", CONFIG)
write_json(run_dir / "proj.json", PROJ)
write_json(run_dir / "label_metadata.json", label_metadata)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(json.dumps(label_metadata, indent=2)[:2000])

result = train_kkt_hardnet(
    model=model,
    X=X,
    Y=Y,
    cfg=make_train_config(CONFIG, PROJ),
    output_dir=run_dir,
    metadata=label_metadata,
)


In [ ]:
summary = {
    "output_dir": result["output_dir"],
    "dims": result["dims"],
    "final": result["final"],
    "component_time_percent": result["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))
